#### npz File LogMel


#### resavanje problema manjka neutrala

In [79]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


# 1. SpecAugment prilagođen za 128 Mel kanala
def spec_augment_mel(
    mel_spec, W=3, freq_mask_max=16, time_mask_max=10, num_masks=1
):
    augmented = mel_spec.copy()
    is_3d = augmented.ndim == 3 and augmented.shape[-1] == 1
    if is_3d:
        augmented = augmented[:, :, 0]

    num_freqs, num_steps = augmented.shape

    # Time Warping
    if num_steps > 2 * W + 1 and W > 0 and np.random.random() < 0.2:
        center = np.random.randint(W, num_steps - W)
        warped_center = center + np.random.randint(-W, W + 1)

        orig_points = np.array([0, center, num_steps - 1])
        warped_points = np.array([0, warped_center, num_steps - 1])

        new_time = np.arange(num_steps)
        src_time = np.interp(new_time, warped_points, orig_points)

        warped_spec = np.zeros_like(augmented)
        for f_idx in range(num_freqs):
            warped_spec[f_idx, :] = np.interp(
                src_time, np.arange(num_steps), augmented[f_idx, :]
            )
        augmented = warped_spec

    # Frequency Masking (veći maks raspon zbog 128 mel opsega)
    if np.random.random() < 0.8:
        f = np.random.randint(1, freq_mask_max)
        f0 = np.random.randint(0, max(1, num_freqs - f))
        augmented[f0 : f0 + f, :] = 0

    # Time Masking
    if np.random.random() < 0.8:
        t = np.random.randint(1, time_mask_max)
        t0 = np.random.randint(0, max(1, num_steps - t))
        augmented[:, t0 : t0 + t] = 0

    if is_3d:
        augmented = np.expand_dims(augmented, axis=-1)

    return augmented


# 2. Učitavanje Log-Mel skupa
print("Učitavanje dataset_mel_spektrogrami.npz...")
data = np.load("/kaggle/input/datasets/lenacvetkovi/logmelspekt/dataset_mel_spektrogrami.npz", allow_pickle=True)
X_all = data["X"]
y_all = data["y"]
speakers_all = data["speakers"]
filenames_all = data["filenames"]

print(f"✓ Učitano: {X_all.shape[0]} uzoraka")
print(f"✓ Shape X: {X_all.shape}")
print(f"✓ Speakers: min={speakers_all.min()}, max={speakers_all.max()}")

unique_speakers = np.unique(speakers_all)
for speaker in sorted(unique_speakers):
    count = np.sum(speakers_all == speaker)
    if 1 <= speaker <= 20:
        split = "TRAIN"
    elif speaker in [21, 22]:
        split = "VAL"
    elif speaker in [23, 24]:
        split = "TEST"
    else:
        split = "UNKNOWN"
    print(f"Speaker {speaker:2d} ({split:6s}): {count:3d} uzoraka")

# 3. Speaker Independent Split i balansiranje
X_tr, y_tr, w_tr = [], [], []
X_va, y_va, w_va = [], [], []
X_te, y_te, w_te = [], [], []

train_speakers = set(range(1, 21))
val_speakers = set([21, 22])
test_speakers = set([23, 24])

for c in range(8):
    indices = np.where(y_all == c)[0]

    train_indices = indices[
        np.isin(speakers_all[indices], list(train_speakers))
    ]
    val_indices = indices[np.isin(speakers_all[indices], list(val_speakers))]
    test_indices = indices[np.isin(speakers_all[indices], list(test_speakers))]

    train_w = np.array(
        [
            (
                2.0
                if len(str(filenames_all[idx]).split("-")) >= 4
                and str(filenames_all[idx]).split("-")[3] == "02"
                else 1.0
            )
            for idx in train_indices
        ],
        dtype=np.float32,
    )

    val_w = np.array(
        [
            (
                2.0
                if len(str(filenames_all[idx]).split("-")) >= 4
                and str(filenames_all[idx]).split("-")[3] == "02"
                else 1.0
            )
            for idx in val_indices
        ],
        dtype=np.float32,
    )

    test_w = np.array(
        [
            (
                2.0
                if len(str(filenames_all[idx]).split("-")) >= 4
                and str(filenames_all[idx]).split("-")[3] == "02"
                else 1.0
            )
            for idx in test_indices
        ],
        dtype=np.float32,
    )

    X_orig = X_all[train_indices]

    if c == 0:
        X_aug = np.array(
            [
                spec_augment_mel(
                    x,
                    W=2,
                    freq_mask_max=16,
                    time_mask_max=5,
                    num_masks=1,
                )
                for x in X_orig
            ]
        )

        X_tr.append(np.concatenate((X_orig, X_aug), axis=0))
        y_tr.append(np.tile(y_all[train_indices], 2))
        w_tr.append(np.tile(train_w, 2))
        print(
            f"Klasa {c} (NEUTRAL): {len(train_indices)} train → {len(train_indices) * 2} sa aug"
        )
    else:
        X_tr.append(X_orig)
        y_tr.append(y_all[train_indices])
        w_tr.append(train_w)
        print(f"Klasa {c}: {len(train_indices)} train uzoraka")

    if len(val_indices) > 0:
        X_va.append(X_all[val_indices])
        y_va.append(y_all[val_indices])
        w_va.append(val_w)
        print(f"  → Val: {len(val_indices)} uzoraka")

    if len(test_indices) > 0:
        X_te.append(X_all[test_indices])
        y_te.append(y_all[test_indices])
        w_te.append(test_w)
        print(f"  → Test: {len(test_indices)} uzoraka")

# Spajanje i permutacija
X_train, y_train, w_train = (
    np.concatenate(X_tr, axis=0),
    np.concatenate(y_tr, axis=0),
    np.concatenate(w_tr, axis=0),
)
train_perm = np.random.permutation(len(y_train))
X_train, y_train, w_train = (
    X_train[train_perm],
    y_train[train_perm],
    w_train[train_perm],
)

X_val, y_val, w_val = (
    np.concatenate(X_va, axis=0),
    np.concatenate(y_va, axis=0),
    np.concatenate(w_va, axis=0),
)
val_perm = np.random.permutation(len(y_val))
X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

X_test, y_test, w_test = (
    np.concatenate(X_te, axis=0),
    np.concatenate(y_te, axis=0),
    np.concatenate(w_te, axis=0),
)
test_perm = np.random.permutation(len(y_test))
X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]

print(f"\n=== FINALNA SPEAKER-INDEPENDENT PODELA ===")
print(f"Train: {X_train.shape[0]} uzoraka (govornici 1-20)")
print(f"Val:   {X_val.shape[0]} uzoraka (govornici 21-22)")
print(f"Test:  {X_test.shape[0]} uzoraka (govornici 23-24)")

# 4. Normalizacija po 128 Mel frekvencijskih kanala
mean = np.mean(X_train, axis=(0, 2, 3), keepdims=True)  # (1, 128, 1, 1)
std = np.std(X_train, axis=(0, 2, 3), keepdims=True)  # (1, 128, 1, 1)

print(f"Mean shape: {mean.shape}, Std shape: {std.shape}")

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)


# 5. PyTorch Dataset & DataLoaders
class AudioDataset(Dataset):
    def __init__(self, X, y, weights):
        if X.ndim == 3:
            X = np.expand_dims(X, axis=1)
        elif X.ndim == 4 and X.shape[-1] == 1:
            X = np.transpose(X, (0, 3, 1, 2))
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


train_loader_logmel = DataLoader(
    AudioDataset(X_train, y_train, w_train), batch_size=256, shuffle=True
)
val_loader_logmel = DataLoader(
    AudioDataset(X_val, y_val, w_val), batch_size=256, shuffle=False
)
test_loader_logmel = DataLoader(
    AudioDataset(X_test, y_test, w_test), batch_size=256, shuffle=False
)

print(f"\n✓ Log-Mel Data loaderi kreirani!")
print(f"  Train loader: {len(train_loader_logmel)} batcheva")
print(f"  Val loader:   {len(val_loader_logmel)} batcheva")
print(f"  Test loader:  {len(test_loader_logmel)} batcheva")

Učitavanje dataset_mel_spektrogrami.npz...
✓ Učitano: 1440 uzoraka
✓ Shape X: (1440, 128, 94, 1)
✓ Speakers: min=1, max=24
Speaker  1 (TRAIN ):  60 uzoraka
Speaker  2 (TRAIN ):  60 uzoraka
Speaker  3 (TRAIN ):  60 uzoraka
Speaker  4 (TRAIN ):  60 uzoraka
Speaker  5 (TRAIN ):  60 uzoraka
Speaker  6 (TRAIN ):  60 uzoraka
Speaker  7 (TRAIN ):  60 uzoraka
Speaker  8 (TRAIN ):  60 uzoraka
Speaker  9 (TRAIN ):  60 uzoraka
Speaker 10 (TRAIN ):  60 uzoraka
Speaker 11 (TRAIN ):  60 uzoraka
Speaker 12 (TRAIN ):  60 uzoraka
Speaker 13 (TRAIN ):  60 uzoraka
Speaker 14 (TRAIN ):  60 uzoraka
Speaker 15 (TRAIN ):  60 uzoraka
Speaker 16 (TRAIN ):  60 uzoraka
Speaker 17 (TRAIN ):  60 uzoraka
Speaker 18 (TRAIN ):  60 uzoraka
Speaker 19 (TRAIN ):  60 uzoraka
Speaker 20 (TRAIN ):  60 uzoraka
Speaker 21 (VAL   ):  60 uzoraka
Speaker 22 (VAL   ):  60 uzoraka
Speaker 23 (TEST  ):  60 uzoraka
Speaker 24 (TEST  ):  60 uzoraka
Klasa 0 (NEUTRAL): 80 train → 160 sa aug
  → Val: 8 uzoraka
  → Test: 8 uzoraka
Klasa

#### 2N

##### DOBAR

In [80]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel2N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_2N_best.pth")


class LOGMEL_2CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_2CNN_Small, self).__init__()
        n = 2
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 2N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = LOGMEL_2CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)

model = model.to(device).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!
Epoha 01/30 | Train Loss: 3.0004 | Val Loss: 2.0803 | Val Acc: 13.33% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.9955 | Val Loss: 2.0795 | Val Acc: 12.50% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.9938 | Val Loss: 2.0781 | Val Acc: 14.17% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.9872 | Val Loss: 2.0768 | Val Acc: 13.33% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.9800 | Val Loss: 2.0759 | Val Acc: 13.33% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.9787 | Val Loss: 2.0748 | Val Acc: 13.33% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.9840 | Val Loss: 2.0737 | Val Acc: 13.33% [Model je sačuvan]
Epoha 08/30 | Train Loss: 2.9776 | Val Loss: 2.0725 | Val Acc: 15.00% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.9766 | Val Loss: 2.0708 | Val Acc: 16.67% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.9748 | Val Loss: 2.0685 | Val Acc: 16.67% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.9699 | Val Loss: 2.0661

In [81]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_2CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel2N/model_logmel_2N_best.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             happy (13.72%)
Vreme pojedinačne inferencije:  1.453 ms

Verovatnoće po klasama:
  neutral   :   9.99%
  calm      :  12.04%
  happy     :  13.72%
  sad       :  13.23%
  angry     :  12.74%
  fearful   :  13.62%
  disgust   :  12.27%
  surprised :  12.39%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.0056 s
Prosečno vreme po uzorku:  0.047 ms
Brzina obrade (throughput): 21259.76 FPS (uzoraka`/s)


#### 4N

In [82]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel4N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_4N_best.pth")


class LOGMEL_4CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_4CNN_Small, self).__init__()
        n = 4
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 2N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = LOGMEL_4CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)

model = model.to(device).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!
Epoha 01/30 | Train Loss: 3.0443 | Val Loss: 2.0876 | Val Acc: 13.33% [Model je sačuvan]
Epoha 02/30 | Train Loss: 3.0325 | Val Loss: 2.0874 | Val Acc: 12.50% [Model je sačuvan]
Epoha 03/30 | Train Loss: 3.0251 | Val Loss: 2.0870 | Val Acc: 10.00% [Model je sačuvan]
Epoha 04/30 | Train Loss: 3.0212 | Val Loss: 2.0862 | Val Acc: 9.17% [Model je sačuvan]
Epoha 05/30 | Train Loss: 3.0087 | Val Loss: 2.0853 | Val Acc: 9.17% [Model je sačuvan]
Epoha 06/30 | Train Loss: 3.0102 | Val Loss: 2.0839 | Val Acc: 9.17% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.9942 | Val Loss: 2.0822 | Val Acc: 9.17% [Model je sačuvan]
Epoha 08/30 | Train Loss: 2.9930 | Val Loss: 2.0803 | Val Acc: 10.00% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.9920 | Val Loss: 2.0786 | Val Acc: 7.50% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.9862 | Val Loss: 2.0769 | Val Acc: 8.33% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.9772 | Val Loss: 2.0751 | Val

In [83]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_4CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel4N/model_logmel_4N_best.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             sad (13.98%)
Vreme pojedinačne inferencije:  1.559 ms

Verovatnoće po klasama:
  neutral   :  10.49%
  calm      :  11.52%
  happy     :  13.81%
  sad       :  13.98%
  angry     :  12.65%
  fearful   :  13.80%
  disgust   :  11.72%
  surprised :  12.03%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.0061 s
Prosečno vreme po uzorku:  0.051 ms
Brzina obrade (throughput): 19607.18 FPS (uzoraka`/s)


#### 8N

In [84]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel8N"
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_8N_best.pth")


class LOGMEL_8CNN_Small(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_8CNN_Small, self).__init__()
        n = 8
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

#m METRIKE 2N

def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    class_metrics = []
    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)

        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )

        class_metrics.append(
            {
                "Emocija": emotion,
                "TP": TP,
                "FP": FP,
                "TN": TN,
                "FN": FN,
                "Hit Rate (%)": round(hit_rate, 2),
                "Precision (%)": round(precision, 2),
                "Class Acc (%)": round(class_acc, 2),
                "F1-Score": round(f1, 4),
            }
        )

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc


emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = LOGMEL_8CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)
model = model.to(device).to(device)

criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}


# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )

        optimizer.zero_grad()
        outputs = model(inputs)

        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)

            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)

            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)

    saved_flag = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    
    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)

        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))


# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

#finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")
with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))




Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!
Epoha 01/30 | Train Loss: 2.9991 | Val Loss: 2.0785 | Val Acc: 13.33% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.9862 | Val Loss: 2.0786 | Val Acc: 13.33%
Epoha 03/30 | Train Loss: 2.9738 | Val Loss: 2.0782 | Val Acc: 13.33% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.9692 | Val Loss: 2.0769 | Val Acc: 13.33% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.9539 | Val Loss: 2.0754 | Val Acc: 13.33% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.9603 | Val Loss: 2.0733 | Val Acc: 13.33% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.9537 | Val Loss: 2.0711 | Val Acc: 15.00% [Model je sačuvan]
Epoha 08/30 | Train Loss: 2.9455 | Val Loss: 2.0687 | Val Acc: 25.83% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.9356 | Val Loss: 2.0656 | Val Acc: 19.17% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.9348 | Val Loss: 2.0623 | Val Acc: 20.00% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.9342 | Val Loss: 2.0582 | Val Acc: 24.17% 

In [85]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_8CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel8N/model_logmel_8N_best.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             happy (15.79%)
Vreme pojedinačne inferencije:  1.431 ms

Verovatnoće po klasama:
  neutral   :  10.93%
  calm      :  10.46%
  happy     :  15.79%
  sad       :  13.25%
  angry     :  14.52%
  fearful   :  12.99%
  disgust   :  13.71%
  surprised :   8.35%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.0095 s
Prosečno vreme po uzorku:  0.079 ms
Brzina obrade (throughput): 12655.54 FPS (uzoraka`/s)


#### 16N

In [86]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel16N" #ovde
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_16N_best.pth") #ovde

class LOGMEL_16CNN_Small(nn.Module): #ovde
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_16CNN_Small, self).__init__() #ovde
        n = 16 #ovde
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# m METRIKE 16N
def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    class_metrics = []

    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)
        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )
        class_metrics.append({
            "Emocija": emotion,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "Hit Rate (%)": round(hit_rate, 2),
            "Precision (%)": round(precision, 2),
            "Class Acc (%)": round(class_acc, 2),
            "F1-Score": round(f1, 4),
        })

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc

emotion_names = [
    "neutral", "calm", "happy", "sad",
    "angry", "fearful", "disgust", "surprised",
]

# inicijalizacija
model = LOGMEL_16CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)

model = model.to(device).to(device) #ovde
criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)
epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}

# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )
        optimizer.zero_grad()
        outputs = model(inputs)
        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples
    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )
    scheduler.step(epoch_val_loss)
    saved_flag = ""

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()
test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)
        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples
cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))

# Test confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)
plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)
plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)
plt.tight_layout()
test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()
print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()
print(f"history: {history_path}")

# finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")

with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))

Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!
Epoha 01/30 | Train Loss: 2.9863 | Val Loss: 2.0781 | Val Acc: 13.33% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.9696 | Val Loss: 2.0775 | Val Acc: 18.33% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.9448 | Val Loss: 2.0763 | Val Acc: 19.17% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.9261 | Val Loss: 2.0744 | Val Acc: 14.17% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.9158 | Val Loss: 2.0717 | Val Acc: 14.17% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.9021 | Val Loss: 2.0681 | Val Acc: 14.17% [Model je sačuvan]
Epoha 07/30 | Train Loss: 2.8821 | Val Loss: 2.0631 | Val Acc: 15.83% [Model je sačuvan]
Epoha 08/30 | Train Loss: 2.8735 | Val Loss: 2.0566 | Val Acc: 16.67% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.8615 | Val Loss: 2.0478 | Val Acc: 17.50% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.8422 | Val Loss: 2.0375 | Val Acc: 22.50% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.8373 | Val Loss: 2.0250

In [87]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_16CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel16N/model_logmel_16N_best.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             calm (23.47%)
Vreme pojedinačne inferencije:  1.474 ms

Verovatnoće po klasama:
  neutral   :  10.02%
  calm      :  23.47%
  happy     :   9.16%
  sad       :  20.58%
  angry     :   7.24%
  fearful   :  10.44%
  disgust   :  10.23%
  surprised :   8.85%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.0222 s
Prosečno vreme po uzorku:  0.185 ms
Brzina obrade (throughput): 5394.54 FPS (uzoraka`/s)


#### 32N

In [88]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel32N" #ovde
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_32N_best.pth") #ovde

class LOGMEL_32CNN_Small(nn.Module): #ovde
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_32CNN_Small, self).__init__() #ovde
        n = 32 #ovde
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# m METRIKE 32N
def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    class_metrics = []

    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)
        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )
        class_metrics.append({
            "Emocija": emotion,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "Hit Rate (%)": round(hit_rate, 2),
            "Precision (%)": round(precision, 2),
            "Class Acc (%)": round(class_acc, 2),
            "F1-Score": round(f1, 4),
        })

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = LOGMEL_32CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)

model = model.to(device).to(device) #ovde
criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)
epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}

# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )
        optimizer.zero_grad()
        outputs = model(inputs)
        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)
    saved_flag = ""

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)
        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples

cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))

# Test confusion matrix
plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)

plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)

plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)

plt.tight_layout()

test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()

print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(
    history["epoch"],
    history["train_loss"],
    label="Train Loss",
    marker="o"
)

plt.plot(
    history["epoch"],
    history["val_loss"],
    label="Val Loss",
    marker="s"
)

plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label=f"Best Epoch ({best_epoch})"
)

plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)

plt.plot(
    history["epoch"],
    history["val_acc"],
    label="Val Accuracy",
    marker="s"
)

plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label=f"Best Epoch ({best_epoch})"
)

plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()

print(f"history: {history_path}")

# finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")

with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))

Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!
Epoha 01/30 | Train Loss: 2.9781 | Val Loss: 2.0755 | Val Acc: 16.67% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.9124 | Val Loss: 2.0744 | Val Acc: 13.33% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.8694 | Val Loss: 2.0726 | Val Acc: 13.33% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.8361 | Val Loss: 2.0706 | Val Acc: 13.33% [Model je sačuvan]
Epoha 05/30 | Train Loss: 2.7952 | Val Loss: 2.0697 | Val Acc: 13.33% [Model je sačuvan]
Epoha 06/30 | Train Loss: 2.7702 | Val Loss: 2.0697 | Val Acc: 13.33%
Epoha 07/30 | Train Loss: 2.7265 | Val Loss: 2.0698 | Val Acc: 15.00%
Epoha 08/30 | Train Loss: 2.6997 | Val Loss: 2.0689 | Val Acc: 22.50% [Model je sačuvan]
Epoha 09/30 | Train Loss: 2.6667 | Val Loss: 2.0660 | Val Acc: 26.67% [Model je sačuvan]
Epoha 10/30 | Train Loss: 2.6267 | Val Loss: 2.0563 | Val Acc: 25.83% [Model je sačuvan]
Epoha 11/30 | Train Loss: 2.5909 | Val Loss: 2.0389 | Val Acc: 25.83% [Model je sačuvan]


In [89]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_32CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel32N/model_logmel_32N_best.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             calm (42.00%)
Vreme pojedinačne inferencije:  1.850 ms

Verovatnoće po klasama:
  neutral   :  11.31%
  calm      :  42.00%
  happy     :   6.64%
  sad       :  23.65%
  angry     :   1.08%
  fearful   :   5.04%
  disgust   :   7.90%
  surprised :   2.37%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.0223 s
Prosečno vreme po uzorku:  0.186 ms
Brzina obrade (throughput): 5390.60 FPS (uzoraka`/s)


#### 64N

In [90]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel64N" #ovde
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_64N_best.pth") #ovde

class LOGMEL_64CNN_Small(nn.Module): #ovde
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_64CNN_Small, self).__init__() #ovde
        n = 64 #ovde
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# m METRIKE 64N
def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    class_metrics = []

    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)
        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )
        class_metrics.append({
            "Emocija": emotion,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "Hit Rate (%)": round(hit_rate, 2),
            "Precision (%)": round(precision, 2),
            "Class Acc (%)": round(class_acc, 2),
            "F1-Score": round(f1, 4),
        })

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = LOGMEL_64CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)

model = model.to(device).to(device) #ovde
criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)
epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}

# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )
        optimizer.zero_grad()
        outputs = model(inputs)
        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)
    saved_flag = ""

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)
        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples

cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))

# Test confusion matrix
plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)

plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)

plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)

plt.tight_layout()

test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()

print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)

plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()

print(f"history: {history_path}")

# finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")

with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))

Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!
Epoha 01/30 | Train Loss: 2.9974 | Val Loss: 2.0788 | Val Acc: 13.33% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.8659 | Val Loss: 2.0751 | Val Acc: 13.33% [Model je sačuvan]
Epoha 03/30 | Train Loss: 2.7937 | Val Loss: 2.0735 | Val Acc: 13.33% [Model je sačuvan]
Epoha 04/30 | Train Loss: 2.7304 | Val Loss: 2.0748 | Val Acc: 13.33%
Epoha 05/30 | Train Loss: 2.6698 | Val Loss: 2.0801 | Val Acc: 14.17%
Epoha 06/30 | Train Loss: 2.6160 | Val Loss: 2.0915 | Val Acc: 16.67%
Epoha 07/30 | Train Loss: 2.5562 | Val Loss: 2.1039 | Val Acc: 23.33%
Epoha 08/30 | Train Loss: 2.5128 | Val Loss: 2.1146 | Val Acc: 25.83%
Epoha 09/30 | Train Loss: 2.4765 | Val Loss: 2.1230 | Val Acc: 25.83%
Epoha 10/30 | Train Loss: 2.4376 | Val Loss: 2.1082 | Val Acc: 25.83%
Epoha 11/30 | Train Loss: 2.4070 | Val Loss: 2.0975 | Val Acc: 27.50%
Epoha 12/30 | Train Loss: 2.3829 | Val Loss: 2.0472 | Val Acc: 28.33% [Model je sačuvan]
Epoha 13/30 | Train Loss:

In [91]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_64CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel64N/model_logmel_64N_best.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 67. fajl u test skupu ===
Stvarna emocija (Ground Truth): sad
Predviđena emocija:             calm (34.60%)
Vreme pojedinačne inferencije:  1.435 ms

Verovatnoće po klasama:
  neutral   :  12.50%
  calm      :  34.60%
  happy     :   7.86%
  sad       :  25.17%
  angry     :   1.64%
  fearful   :   7.80%
  disgust   :   6.77%
  surprised :   3.65%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.0415 s
Prosečno vreme po uzorku:  0.345 ms
Brzina obrade (throughput): 2894.39 FPS (uzoraka`/s)


#### 128N

In [92]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel128N" #ovde
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_128N_best.pth") #ovde

class LOGMEL_128CNN_Small(nn.Module): #ovde
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_128CNN_Small, self).__init__() #ovde
        n = 128 #ovde
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# m METRIKE 128N
def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    class_metrics = []

    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)
        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )
        class_metrics.append({
            "Emocija": emotion,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "Hit Rate (%)": round(hit_rate, 2),
            "Precision (%)": round(precision, 2),
            "Class Acc (%)": round(class_acc, 2),
            "F1-Score": round(f1, 4),
        })

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = LOGMEL_128CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)

model = model.to(device).to(device) #ovde
criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)
epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}

# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )
        optimizer.zero_grad()
        outputs = model(inputs)
        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)
    saved_flag = ""

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)
        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples

cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))

# Test confusion matrix
plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)

plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)

plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)

plt.tight_layout()

test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()

print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)

plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()

print(f"history: {history_path}")

# finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")

with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))

Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!
Epoha 01/30 | Train Loss: 2.9679 | Val Loss: 2.0724 | Val Acc: 20.00% [Model je sačuvan]
Epoha 02/30 | Train Loss: 2.7629 | Val Loss: 2.0766 | Val Acc: 13.33%
Epoha 03/30 | Train Loss: 2.6240 | Val Loss: 2.0811 | Val Acc: 13.33%
Epoha 04/30 | Train Loss: 2.5164 | Val Loss: 2.0874 | Val Acc: 14.17%
Epoha 05/30 | Train Loss: 2.4128 | Val Loss: 2.1226 | Val Acc: 25.83%
Epoha 06/30 | Train Loss: 2.3204 | Val Loss: 2.1708 | Val Acc: 25.83%
Epoha 07/30 | Train Loss: 2.2571 | Val Loss: 2.2114 | Val Acc: 25.83%
Epoha 08/30 | Train Loss: 2.2027 | Val Loss: 2.2169 | Val Acc: 25.83%
Epoha 09/30 | Train Loss: 2.1615 | Val Loss: 2.3067 | Val Acc: 25.83%
Epoha 10/30 | Train Loss: 2.1177 | Val Loss: 2.2768 | Val Acc: 27.50%
Epoha 11/30 | Train Loss: 2.0880 | Val Loss: 2.2798 | Val Acc: 29.17%
Epoha 12/30 | Train Loss: 2.0607 | Val Loss: 2.2129 | Val Acc: 30.83%
Epoha 13/30 | Train Loss: 2.0442 | Val Loss: 2.2308 | Val Acc: 30.83%
Epoha 14/30 

In [93]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_128CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel128N/model_logmel_128N_best.pth", map_location=device))
model.eval()

sample_idx = 64
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

=== Predikcija za 65. fajl u test skupu ===
Stvarna emocija (Ground Truth): fearful
Predviđena emocija:             sad (26.39%)
Vreme pojedinačne inferencije:  1.420 ms

Verovatnoće po klasama:
  neutral   :   9.67%
  calm      :  20.49%
  happy     :  14.69%
  sad       :  26.39%
  angry     :   1.99%
  fearful   :  18.33%
  disgust   :   5.43%
  surprised :   3.01%


=== Benchmark na celom test skupu ===
Ukupno testirano uzoraka: 120
Ukupno trajanje:           0.1009 s
Prosečno vreme po uzorku:  0.841 ms
Brzina obrade (throughput): 1189.42 FPS (uzoraka`/s)


#### 256N

In [94]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)
print(f"Koristi se uređaj: {device}")

save_dir = "logmel256N" #ovde
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, "model_logmel_256N_best.pth") #ovde

class LOGMEL_256CNN_Small(nn.Module): #ovde
    def __init__(self, num_classes=8, dropout_rate=0.2):
        super(LOGMEL_256CNN_Small, self).__init__() #ovde
        n = 256 #ovde
        self.features = nn.Sequential(
            nn.Conv2d(1, n, kernel_size=3, padding=1),
            nn.BatchNorm2d(n),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n, n*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(n*4),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        k = 64 if n*4 < 64 else n*8
        self.classifier = nn.Sequential(
            nn.Linear(n*4, k),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(k, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# m METRIKE 128N
def compute_epoch_metrics(y_true, y_pred, emotion_names):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    num_classes = len(emotion_names)
    total_samples = len(y_true)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    class_metrics = []

    for i, emotion in enumerate(emotion_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = total_samples - (TP + FP + FN)
        hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
        precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
        class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
        f1 = (
            2 * (precision * hit_rate) / (precision + hit_rate) / 100
            if (precision + hit_rate) > 0
            else 0.0
        )
        class_metrics.append({
            "Emocija": emotion,
            "TP": TP,
            "FP": FP,
            "TN": TN,
            "FN": FN,
            "Hit Rate (%)": round(hit_rate, 2),
            "Precision (%)": round(precision, 2),
            "Class Acc (%)": round(class_acc, 2),
            "F1-Score": round(f1, 4),
        })

    df_metrics = pd.DataFrame(class_metrics)
    overall_acc = (np.trace(cm) / total_samples) * 100
    return cm, df_metrics, overall_acc

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]

# inicijalizacija
model = LOGMEL_256CNN_Small(num_classes=len(emotion_names), dropout_rate=0.2)

# OVDJE IDE PARALELIZACIJA (nakon što je model kreiran)
if torch.cuda.device_count() > 1:
    print(f"CUDA dostupna! Koristi se {torch.cuda.device_count()} GPU-a!")
    model = nn.DataParallel(model)

model = model.to(device).to(device) #ovde
criterion_train = nn.CrossEntropyLoss(reduction="none")
criterion_eval = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00027, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)
epochs = 30
best_val_loss = float("inf")
best_epoch = 0

# History za pleotiranje
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
}

# treniranjw
for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0
    total_train_samples = 0

    for inputs, labels, weights in train_loader_logmel:
        inputs, labels, weights = (
            inputs.to(device),
            labels.to(device),
            weights.to(device),
        )
        optimizer.zero_grad()
        outputs = model(inputs)
        unweighted_loss = criterion_train(outputs, labels)
        weighted_loss = unweighted_loss * weights
        loss = weighted_loss.mean()
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)
        total_train_samples += inputs.size(0)

    epoch_train_loss = running_train_loss / total_train_samples

    model.eval()
    val_y_true, val_y_pred = [], []
    running_val_loss = 0.0
    total_val_samples = 0

    with torch.no_grad():
        for inputs, labels, _ in val_loader_logmel:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion_eval(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)
            total_val_samples += inputs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_y_true.extend(labels.cpu().numpy())
            val_y_pred.extend(preds)

    epoch_val_loss = running_val_loss / total_val_samples
    cm_val, df_metrics, val_acc = compute_epoch_metrics(
        val_y_true, val_y_pred, emotion_names
    )

    scheduler.step(epoch_val_loss)
    saved_flag = ""

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch
        
        # Pravilno čuvanje kada se koristi DataParallel
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), best_model_path)
        else:
            torch.save(model.state_dict(), best_model_path)
            
        saved_flag = " [Model je sačuvan]"

    history["epoch"].append(epoch)
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoha {epoch:02d}/{epochs:02d} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved_flag}"
    )

print(f"\n")
print(f"Najbolja epoha: {best_epoch} sa Val Loss: {best_val_loss:.4f}")
print(f"\n")

# testiranje aksli na testu
print("najbolji rez...")
state_dict = torch.load(best_model_path)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_y_true, test_y_pred = [], []
running_test_loss = 0.0
total_test_samples = 0

with torch.no_grad():
    for inputs, labels, _ in test_loader_logmel:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion_eval(outputs, labels)
        running_test_loss += loss.item() * inputs.size(0)
        total_test_samples += inputs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        test_y_true.extend(labels.cpu().numpy())
        test_y_pred.extend(preds)

epoch_test_loss = running_test_loss / total_test_samples

cm_test, df_test_metrics, test_acc = compute_epoch_metrics(
    test_y_true, test_y_pred, emotion_names
)

print(f"\n")
print(f"FINALNI TEST REZULTATI (Najbolji Model - Epoha {best_epoch})")
print(f"\n")
print(f"Test Loss: {epoch_test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%\n")
print("Test Metrrike po klasi:")
print(df_test_metrics.to_string(index=False))

# Test confusion matrix
plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_test,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=emotion_names,
    yticklabels=emotion_names,
)

plt.xlabel("Predviđena emocija", fontsize=12)
plt.ylabel("Stvarna emocija", fontsize=12)

plt.title(
    f"FINALNA TEST MATRICA KONFUZIJE\nTest Accuracy: {test_acc:.2f}% (Epoha {best_epoch})",
    fontsize=14,
)

plt.tight_layout()

test_cm_path = os.path.join(save_dir, "FINAL_test_confusion_matrix.png")
plt.savefig(test_cm_path, dpi=300)
plt.close()

print(f"\nconfusion matrix: {test_cm_path}")

# Test metrrike CSV
test_metrics_csv = os.path.join(save_dir, "FINAL_test_metrics.csv")
df_test_metrics.to_csv(test_metrics_csv, index=False)
print(f"metrike u: {test_metrics_csv}")

# Training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(history["epoch"], history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.title("Loss Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)

plt.plot(history["epoch"], history["val_acc"], label="Val Accuracy", marker="s")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"Best Epoch ({best_epoch})")
plt.xlabel("Epoha")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy Kriva")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

history_path = os.path.join(save_dir, "training_history.png")
plt.savefig(history_path, dpi=300)
plt.close()

print(f"history: {history_path}")

# finalan model
summary_txt = os.path.join(save_dir, "FINAL_RESULTS.txt")

with open(summary_txt, "w", encoding="utf-8") as f:
    f.write(f"FINALNI REZULTATI TRENIRANJA\n")
    f.write(f"\n\n")
    f.write(f"Najbolja epoha: {best_epoch}\n")
    f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
    f.write(f"Best Validation Accuracy: {history['val_acc'][best_epoch-1]:.2f}%\n\n")
    f.write(f"TEST REZULTATI:\n")
    f.write(f"Test Loss: {epoch_test_loss:.4f}\n")
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write(f"TEST METRRIKE PO KLASI:\n")
    f.write(df_test_metrics.to_string(index=False))

Koristi se uređaj: cuda
CUDA dostupna! Koristi se 2 GPU-a!


OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/2287898318.py", line 49, in forward
    x = self.features(x)
        ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/

In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_256CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("/kaggle/working/logmel256N/model_logmel_256N_best.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")

#### 512N

In [ ]:
# import os
# import librosa
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# import seaborn as sns
# from sklearn.metrics import confusion_matrix
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader, Dataset


# def spec_augment(mel_spec, freq_mask_max=8, time_mask_max=12):
#     augmented = mel_spec.copy()
    
#     is_3d = (augmented.ndim == 3 and augmented.shape[-1] == 1)
#     if is_3d:
#         augmented = augmented[:, :, 0]
        
#     num_freqs, num_steps = augmented.shape
    
#     # Frequency Masking
#     f = np.random.randint(1, freq_mask_max)
#     f0 = np.random.randint(0, max(1, num_freqs - f))
#     augmented[f0:f0+f, :] = 0
    
#     # Time Masking
#     t = np.random.randint(1, time_mask_max)
#     t0 = np.random.randint(0, max(1, num_steps - t))
#     augmented[:, t0:t0+t] = 0
    
#     if is_3d:
#         augmented = np.expand_dims(augmented, axis=-1)
        
#     return augmented

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# np.random.seed(42)
# torch.manual_seed(42)
# print(f"Koristi se uređaj: {device}")

# save_dir = "logmel512N"
# os.makedirs(save_dir, exist_ok=True)

# best_model_path = "model_logmel_512N.pth"

# data = np.load("dataset_mel_spektrogrami.npz")
# X_all, y_all = data["X"], data["y"]
# filenames_all = data["filenames"] if "filenames" in data else None

# X_tr, y_tr, w_tr = [], [], []
# X_va, y_va, w_va = [], [], []
# X_te, y_te, w_te = [], [], []

# for c in range(8):
#     indices = np.where(y_all == c)[0]
#     np.random.shuffle(indices)

#     weights_c = []
#     for idx in indices:
#         if filenames_all is not None:
#             fname = os.path.basename(str(filenames_all[idx]))
#             parts = fname.split("-")
#             if len(parts) >= 4 and parts[3] == "02":
#                 weights_c.append(2.0)
#             else:
#                 weights_c.append(1.0)
#         else:
#             weights_c.append(1.0)
#     weights_c = np.array(weights_c, dtype=np.float32)

#     if c == 0: 
#         train_idx = indices[:64]
#         val_idx = indices[64:80]
#         test_idx = indices[80:96]

#         train_w = weights_c[:64]

#         # Augmentacija neutralne klase pomoću SpecAugment-a
#         X_orig = X_all[train_idx]
#         X_aug = np.array([spec_augment(x) for x in X_orig])
        
#         X_train_neutral = np.concatenate((X_orig, X_aug), axis=0)
#         y_train_neutral = np.tile(y_all[train_idx], 2)
#         w_train_neutral = np.tile(train_w, 2)

#         X_tr.append(X_train_neutral)
#         y_tr.append(y_train_neutral)
#         w_tr.append(w_train_neutral)

#         X_va.append(X_all[val_idx])
#         y_va.append(y_all[val_idx])
#         w_va.append(weights_c[64:80])

#         X_te.append(X_all[test_idx])
#         y_te.append(y_all[test_idx])
#         w_te.append(weights_c[80:96])
        

#     else:  
#         train_idx = indices[:128]
#         val_idx = indices[128:160]
#         test_idx = indices[160:192]

#         X_tr.append(X_all[train_idx])
#         y_tr.append(y_all[train_idx])
#         w_tr.append(weights_c[:128])

#         X_va.append(X_all[val_idx])
#         y_va.append(y_all[val_idx])
#         w_va.append(weights_c[128:160])

#         X_te.append(X_all[test_idx])
#         y_te.append(y_all[test_idx])
#         w_te.append(weights_c[160:192])

# X_train, y_train, w_train = (
#     np.concatenate(X_tr, axis=0),
#     np.concatenate(y_tr, axis=0),
#     np.concatenate(w_tr, axis=0),
# )
# train_perm = np.random.permutation(len(y_train))
# X_train, y_train, w_train = (
#     X_train[train_perm],
#     y_train[train_perm],
#     w_train[train_perm],
# )

# X_val, y_val, w_val = (
#     np.concatenate(X_va, axis=0),
#     np.concatenate(y_va, axis=0),
#     np.concatenate(w_va, axis=0),
# )
# val_perm = np.random.permutation(len(y_val))
# X_val, y_val, w_val = X_val[val_perm], y_val[val_perm], w_val[val_perm]

# X_test, y_test, w_test = (
#     np.concatenate(X_te, axis=0),
#     np.concatenate(y_te, axis=0),
#     np.concatenate(w_te, axis=0),
# )
# test_perm = np.random.permutation(len(y_test))
# X_test, y_test, w_test = X_test[test_perm], y_test[test_perm], w_test[test_perm]


# class AudioDataset(Dataset):
#     def __init__(self, X, y, weights):
#         if X.ndim == 3:
#             X = np.expand_dims(X, axis=1)
#         elif X.ndim == 4 and X.shape[-1] == 1:
#             X = np.transpose(X, (0, 3, 1, 2))
#         self.X = torch.tensor(X, dtype=torch.float32)
#         self.y = torch.tensor(y, dtype=torch.long)
#         self.weights = torch.tensor(weights, dtype=torch.float32)

#     def __len__(self):
#         return len(self.y)

#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx], self.weights[idx]


# train_loader_logmel = DataLoader(
#     AudioDataset(X_train, y_train, w_train), batch_size=32, shuffle=True
# )
# val_loader_logmel = DataLoader(
#     AudioDataset(X_val, y_val, w_val), batch_size=32, shuffle=False
# )
# test_loader_logmel = DataLoader(
#     AudioDataset(X_test, y_test, w_test), batch_size=32, shuffle=False
# )


# class LOGMEL_512CNN_Small(nn.Module):
#     def __init__(self, num_classes=8, dropout_rate=0.3):
#         super(LOGMEL_512CNN_Small, self).__init__()
#         n=512
#         self.features = nn.Sequential(
#             nn.Conv2d(1, n, kernel_size=3, padding=1),
#             nn.BatchNorm2d(n),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),
#             nn.Conv2d(n, n*2, kernel_size=3, padding=1),
#             nn.BatchNorm2d(n*2),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),
#             nn.Conv2d(n*2, n*4, kernel_size=3, padding=1),
#             nn.BatchNorm2d(n*4),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),
#         )
#         self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
#         if n*4 < 64:
#             k=64
#         else:
#             k=n*8
#         self.classifier = nn.Sequential(
#             nn.Linear(n*4, k),
#             nn.ReLU(),
#             nn.Dropout(p=dropout_rate),
#             nn.Linear(k, num_classes),
#         )

#     def forward(self, x):
#         x = self.features(x)
#         x = self.global_pool(x)
#         x = torch.flatten(x, 1)
#         return self.classifier(x)


# def compute_epoch_metrics(y_true, y_pred, emotion_names):
#     y_true = np.array(y_true)
#     y_pred = np.array(y_pred)
#     num_classes = len(emotion_names)
#     total_samples = len(y_true)

#     cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

#     class_metrics = []
#     for i, emotion in enumerate(emotion_names):
#         TP = cm[i, i]
#         FN = np.sum(cm[i, :]) - TP
#         FP = np.sum(cm[:, i]) - TP
#         TN = total_samples - (TP + FP + FN)

#         hit_rate = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0.0
#         precision = (TP / (TP + FP)) * 100 if (TP + FP) > 0 else 0.0
#         class_acc = ((TP + TN) / total_samples) * 100 if total_samples > 0 else 0.0
#         f1 = (
#             2 * (precision * hit_rate) / (precision + hit_rate) / 100
#             if (precision + hit_rate) > 0
#             else 0.0
#         )

#         class_metrics.append(
#             {
#                 "Emocija": emotion,
#                 "TP": TP,
#                 "FP": FP,
#                 "TN": TN,
#                 "FN": FN,
#                 "Hit Rate (%)": round(hit_rate, 2),
#                 "Precision (%)": round(precision, 2),
#                 "Class Acc (%)": round(class_acc, 2),
#                 "F1-Score": round(f1, 4),
#             }
#         )

#     df_metrics = pd.DataFrame(class_metrics)
#     overall_acc = (np.trace(cm) / total_samples) * 100
#     return cm, df_metrics, overall_acc


# emotion_names = [
#     "neutral",
#     "calm",
#     "happy",
#     "sad",
#     "angry",
#     "fearful",
#     "disgust",
#     "surprised",
# ]

# model = LOGMEL_512CNN_Small(num_classes=len(emotion_names), dropout_rate=0.3).to(device)

# criterion_train = nn.CrossEntropyLoss(reduction="none")
# criterion_eval = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.00027)

# epochs = 30
# best_val_loss = float("inf")

# for epoch in range(1, epochs + 1):
#     model.train()
#     running_train_loss = 0.0
#     total_train_samples = 0

#     for inputs, labels, weights in train_loader_logmel:
#         inputs, labels, weights = (
#             inputs.to(device),
#             labels.to(device),
#             weights.to(device),
#         )

#         optimizer.zero_grad()
#         outputs = model(inputs)

#         unweighted_loss = criterion_train(outputs, labels)
#         weighted_loss = unweighted_loss * weights
#         loss = weighted_loss.mean()

#         loss.backward()
#         optimizer.step()

#         running_train_loss += loss.item() * inputs.size(0)
#         total_train_samples += inputs.size(0)

#     epoch_train_loss = running_train_loss / total_train_samples

#     # --- VALIDACIJA ---
#     model.eval()
#     val_y_true, val_y_pred = [], []
#     running_val_loss = 0.0
#     total_val_samples = 0

#     with torch.no_grad():
#         for inputs, labels, _ in val_loader_logmel:
#             inputs, labels = inputs.to(device), labels.to(device)
#             outputs = model(inputs)
#             loss = criterion_eval(outputs, labels)

#             running_val_loss += loss.item() * inputs.size(0)
#             total_val_samples += inputs.size(0)

#             preds = outputs.argmax(dim=1).cpu().numpy()
#             val_y_true.extend(labels.cpu().numpy())
#             val_y_pred.extend(preds)

#     epoch_val_loss = running_val_loss / total_val_samples
#     cm_val, df_metrics, val_acc = compute_epoch_metrics(
#         val_y_true, val_y_pred, emotion_names
#     )

#     saved_flag = ""
#     if epoch_val_loss < best_val_loss:
#         best_val_loss = epoch_val_loss
#         torch.save(model.state_dict(), best_model_path)
#         saved_flag = f" [Model sačuvan -> {best_model_path}]"

#     # --- TESTIRANJE ---
#     test_y_true, test_y_pred = [], []
#     running_test_loss = 0.0
#     total_test_samples = 0

#     with torch.no_grad():
#         for inputs, labels, _ in test_loader_logmel:
#             inputs, labels = inputs.to(device), labels.to(device)
#             outputs = model(inputs)
#             loss = criterion_eval(outputs, labels)

#             running_test_loss += loss.item() * inputs.size(0)
#             total_test_samples += inputs.size(0)

#             preds = outputs.argmax(dim=1).cpu().numpy()
#             test_y_true.extend(labels.cpu().numpy())
#             test_y_pred.extend(preds)

#     epoch_test_loss = running_test_loss / total_test_samples
#     cm_test, _, test_acc = compute_epoch_metrics(
#         test_y_true, test_y_pred, emotion_names
#     )

#     csv_path = os.path.join(save_dir, f"epoch_{epoch:02d}_metrics.csv")
#     df_metrics.to_csv(csv_path, index=False)

#     summary_txt_path = os.path.join(save_dir, f"epoch_{epoch:02d}_summary.txt")
#     with open(summary_txt_path, "w", encoding="utf-8") as f:
#         f.write(f"Epoha: {epoch}\n")
#         f.write(f"Train Loss: {epoch_train_loss:.4f}\n")
#         f.write(
#             f"Val Loss: {epoch_val_loss:.4f} | Val Accuracy: {val_acc:.2f}%\n"
#         )
#         f.write(
#             f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_acc:.2f}%\n"
#         )

#     plt.figure(figsize=(8, 6))
#     sns.heatmap(
#         cm_val,
#         annot=True,
#         fmt="d",
#         cmap="Blues",
#         xticklabels=emotion_names,
#         yticklabels=emotion_names,
#     )
#     plt.xlabel("Predviđena emocija")
#     plt.ylabel("Stvarna emocija")
#     plt.title(
#         f"Val Matrica Konfuzije - Epoha {epoch:02d}\nVal Tačnost: {val_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}"
#     )
#     plt.tight_layout()

#     cm_path = os.path.join(save_dir, f"epoch_{epoch:02d}_confusion_matrix.png")
#     plt.savefig(cm_path, dpi=300)
#     plt.close()

#     print(
#         f"Epoha {epoch:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.2f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {test_acc:.2f}%{saved_flag}"
#     )





In [ ]:
import time
import torch

emotion_names = [
    "neutral",
    "calm",
    "happy",
    "sad",
    "angry",
    "fearful",
    "disgust",
    "surprised",
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LOGMEL_512CNN_Small(num_classes=len(emotion_names)).to(device)
model.load_state_dict(torch.load("model_logmel_512N.pth", map_location=device))
model.eval()

sample_idx = 66
x_sample, y_true_idx, _ = test_loader_logmel.dataset[sample_idx]

inputs = x_sample.unsqueeze(0).to(device)


if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.no_grad():
    outputs = model(inputs)
    probabilities = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probabilities).item()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed_ms = (time.perf_counter() - start_time) * 1000

predicted_emotion = emotion_names[pred_idx]
true_idx_val = (
    y_true_idx.item()
    if isinstance(y_true_idx, torch.Tensor)
    else y_true_idx
)
true_emotion = emotion_names[true_idx_val]
confidence = probabilities[pred_idx].item() * 100


print(f"=== Predikcija za {sample_idx + 1}. fajl u test skupu ===")
print(f"Stvarna emocija (Ground Truth): {true_emotion}")
print(f"Predviđena emocija:             {predicted_emotion} ({confidence:.2f}%)")
print(f"Vreme pojedinačne inferencije:  {elapsed_ms:.3f} ms\n")

print("Verovatnoće po klasama:")
for emotion, prob in zip(emotion_names, probabilities):
    print(f"  {emotion:10s}: {prob.item() * 100:6.2f}%")

print("\n" + "=" * 50 + "\n")

total_samples = 0
start_time_all = time.perf_counter()

with torch.no_grad():
    for batch in test_loader_logmel:
        batch_inputs = batch[0].to(device)
        batch_size = batch_inputs.size(0)

        _ = model(batch_inputs)
        total_samples += batch_size

if device.type == "cuda":
    torch.cuda.synchronize()

total_time_sec = time.perf_counter() - start_time_all
avg_time_ms = (total_time_sec / total_samples) * 1000
fps = total_samples / total_time_sec

print("=== Benchmark na celom test skupu ===")
print(f"Ukupno testirano uzoraka: {total_samples}")
print(f"Ukupno trajanje:           {total_time_sec:.4f} s")
print(f"Prosečno vreme po uzorku:  {avg_time_ms:.3f} ms")
print(f"Brzina obrade (throughput): {fps:.2f} FPS (uzoraka`/s)")